<a href="https://colab.research.google.com/github/JuliBakagianni/delta_meth/blob/main/run_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sentence_transformers
import transformers
import torch
import sklearn
import yaml

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())

torch version: 2.10.0+cu128
cuda available: True
Device count: 1


---

4) Clone repository & checkout branch

Provide a `GITHUB_URL` variable to clone the repository into `/content/delta_meth`. Uncomment and set your GitHub URL as needed.

---

In [2]:
import os
GITHUB_URL = 'https://github.com/JuliBakagianni/delta_meth.git'
print('Cloning from', GITHUB_URL)
os.system('rm -rf /content/delta_meth')
os.system(f'git clone {GITHUB_URL} /content/delta_meth')

Cloning from https://github.com/JuliBakagianni/delta_meth.git


0

---

7) Write pipeline configuration (YAML/JSON)

Load and print the active configuration from `configs/config.yaml` in the cloned repo. If the file is not present, show a default config and write it to disk.

---

In [3]:
# Load config
import yaml
from pathlib import Path

cfg_path = Path('/content/delta_meth/configs/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
print('Loaded config:')
print(cfg)

Loaded config:
{'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'nli_model': 'joeddav/xlm-roberta-large-xnli', 'chunk_size': 150, 'similarity_threshold': 0.5, 'nli_threshold': 0.7, 'llm_model': 'openai/gpt-4o-mini'}


---

8) Implement pipeline runner and mock fallback

This cell imports the pipeline functions and attempts to run them. If heavy model imports fail, the notebook falls back to a mock runner that simulates embeddings and NLI labels for demonstration.

---

In [4]:
# Run pipeline with fallback
import json
from pathlib import Path

# Ensure repo path is on sys.path
import sys
repo_path = '/content/delta_meth'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

mock_mode = False

# Try importing the real pipeline
from src.pipeline.run_pipeline import run_pipeline
from src.alignment.embeddings import encode_chunks
from src.nli.nli_model import predict_nli
print('Pipeline modules imported successfully; running real pipeline...')
contradiction, detailed = run_pipeline(config_path='/content/delta_meth/configs/config.yaml', verbose=True)
print(f"contradiction: {contradiction}")
if mock_mode:
    # Minimal mock implementation using local functions to simulate outputs
    from src.preprocessing.chunking import chunk_notes
    a = Path('/content/delta_meth/data/raw/dummy_note_20260212.txt')
    b = Path('/content/delta_meth/data/raw/dummy_note_20260213.txt')
    note_a = a.read_text(encoding='utf-8') if a.exists() else 'Patient no fever.'
    note_b = b.read_text(encoding='utf-8') if b.exists() else 'Patient has fever.'
    chunks_a = chunk_notes(note_a, chunk_size=50)
    chunks_b = chunk_notes(note_b, chunk_size=50)
    # Simulate alignment: pair each chunk i with j=i
    detailed = []
    for i, ca in enumerate(chunks_a):
        if i < len(chunks_b):
            cb = chunks_b[i]
        else:
            cb = chunks_b[-1] if chunks_b else ''
        # Mock sim score and NLI: if Greek words differ contain 'πυρετό' vs 'δεν' set contradiction
        sim_score = 0.6
        if 'πυρετό' in ca or 'πυρετό' in cb:
            label = 'contradiction'
            conf = 0.95
        else:
            label = 'neutral'
            conf = 0.6
        entry = {
            'i': i,
            'j': i if i < len(chunks_b) else len(chunks_b)-1,
            'sim_score': sim_score,
            'nli_label': label,
            'nli_confidence': conf,
            'chunk1': ca,
            'chunk2': cb,
        }
        detailed.append(entry)
    # Select highest-confidence contradiction
    contradictions = [d for d in detailed if d['nli_label']=='contradiction']
    contradiction = max(contradictions, key=lambda x: x['nli_confidence']) if contradictions else None

# Save results to results/run_output.json
out = {'contradiction': contradiction, 'detailed_pairs': detailed}
out_path = Path('/content/delta_meth/results')
out_path.mkdir(parents=True, exist_ok=True)
(out_path / 'run_output.json').write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')

print('\nSaved run_output.json to /content/delta_meth/results/run_output.json')
print('Contradiction:', contradiction)


Pipeline modules imported successfully; running real pipeline...
[pipeline] note A -> 1 chunks
[pipeline] note B -> 1 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pipeline] pair (0,0) sim=0.843 nli=contradiction (1.000)
  A: Patient reports no chest pain and is ambulatory. Ο ασθενής δεν έχει πυρετό. Blood pressure stable.
  B: Patient complains of severe chest pain and is non-ambulatory. Ο ασθενής έχει πυρετό. Blood pressure slightly elevated.
[pipeline] contradiction FOUND:
  sim_score=0.843
  nli_confidence=1.000
  chunk A:
    Patient reports no chest pain and is ambulatory. Ο ασθενής δεν έχει πυρετό. Blood pressure stable.
  chunk B:
    Patient complains of severe chest pain and is non-ambulatory. Ο ασθενής έχει πυρετό. Blood pressure slightly elevated.
contradiction: {'i': 0, 'j': 0, 'sim_score': 0.8434746265411377, 'nli_confidence': 0.999645471572876, 'chunk1': 'Patient reports no chest pain and is ambulatory. Ο ασθενής δεν έχει πυρετό. Blood pressure stable.', 'chunk2': 'Patient complains of severe chest pain and is non-ambulatory. Ο ασθενής έχει πυρετό. Blood pressure slightly elevated.'}

Saved run_output.json to /content/delta_meth/r

---

### Load segments.zip and unzip into the repo

Use the cell below to upload a `segments.zip` file (interactive) or mount Drive and copy it. The zip should contain the per-note JSON files under its root (i.e. `note_id.json`).

In [ ]:
# Upload or copy segments.zip into the notebook and extract to data/processed/segments
from pathlib import Path
import zipfile, os
repo_path = '/content/delta_meth'
# clone repo if not present
if not Path(repo_path).exists():
    print('Cloning repository...')
    os.system(f'git clone https://github.com/JuliBakagianni/delta_meth.git {repo_path}')
# interactive upload if segments.zip not present
if not Path('segments.zip').exists():
    try:
        from google.colab import files
        print('Please upload segments.zip (select file in the dialog)')
        uploaded = files.upload()
        for fn, data in uploaded.items():
            open(fn, 'wb').write(data)
    except Exception as e:
        print('Upload skipped or not available:', e)
# ensure target dir exists and extract if zip present
target = Path(repo_path) / 'data' / 'processed' / 'segments'
target.mkdir(parents=True, exist_ok=True)
if Path('segments.zip').exists():
    print('Extracting segments.zip to', target)
    with zipfile.ZipFile('segments.zip', 'r') as z:
        z.extractall(target)
    print('Extraction complete. Files:')
    print(sorted([p.name for p in target.glob('*.json')])[:10])
else:
    print('No segments.zip found; ensure JSONs are placed in', target)


---

### Call pipeline functions interactively

This cell imports `run_pipeline` and runs sequential comparisons for a single note's segments so you can inspect intermediate outputs. If model loading fails, a simple mock fallback will be used for demonstration.

In [ ]:
import sys, json, glob
from pathlib import Path
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.run_pipeline import run_pipeline
# list available segment JSONs
seg_dir = Path(repo_path) / 'data' / 'processed' / 'segments'
files = sorted([p for p in seg_dir.glob('*.json')]) if seg_dir.exists() else []
print('Found', len(files), 'segment files')
# pick first multi-segment file for interactive inspection
sample = None
for p in files:
    data = json.loads(p.read_text(encoding='utf-8'))
    if len(data.get('segments', [])) > 1:
        sample = p
        break
if not sample:
    print('No multi-segment file found; add JSONs to', seg_dir)
else:
    print('Using sample file:', sample)
    note = json.loads(sample.read_text(encoding='utf-8'))
    segs = note.get('segments', [])
    print('Segment count:', len(segs))
    # sequentially compare segment[i] vs segment[i+1]
    for i in range(len(segs)-1):
        a = segs[i].get('text','')
        b = segs[i+1].get('text','')
        print('
--- Comparison', i, 'dates:', segs[i].get('date'), 'vs', segs[i+1].get('date'))
        try:
            contradiction, detailed = run_pipeline(note_a=a, note_b=b, config_path=str(Path(repo_path)/'configs'/'config.yaml'), verbose=True, chunk_unit='sentences', chunk_size=1)
            print('Contradiction:', contradiction)
        except Exception as e:
            print('Real pipeline failed:', e)
            # Mock fallback: simple heuristic demo
            from src.preprocessing.chunking import chunk_notes
            ca = chunk_notes(a, chunk_size=1, chunk_unit='sentences')
            cb = chunk_notes(b, chunk_size=1, chunk_unit='sentences')
            detailed = []
            for idx, sent in enumerate(ca):
                other = cb[idx] if idx < len(cb) else (cb[-1] if cb else '')
                label = 'contradiction' if ('πυρετό' in sent and 'πυρετό' not in other) or ('πυρετό' in other and 'πυρετό' not in sent) else 'neutral'
                detailed.append({'i': idx, 'j': idx if idx < len(cb) else len(cb)-1, 'nli_label': label, 'chunk1': sent, 'chunk2': other})
            contradictions = [d for d in detailed if d['nli_label']=='contradiction']
            contradiction = contradictions[0] if contradictions else None
            print('Mock contradiction:', contradiction)


---

### Batch run (optional)

If you want to run the diagnostic comparisons across many files, uncomment and run the cell below. Beware: this will load models for each comparison and may be slow or require more RAM/GPU.

In [ ]:
# Example batch loop (commented out by default)
# from pathlib import Path
# repo_path = '/content/delta_meth'
# seg_dir = Path(repo_path)/'data'/'processed'/'segments'
# out_dir = Path(repo_path)/'data'/'results'/'diagnostic_shifts'
# out_dir.mkdir(parents=True, exist_ok=True)
# files = sorted([p for p in seg_dir.glob('*.json')])
# for p in files[:50]:  # limit to first 50 for demo
#     note = json.loads(p.read_text(encoding='utf-8'))
#     segs = note.get('segments', [])
#     if len(segs)<=1:
#         continue
#     results = []
#     for i in range(len(segs)-1):
#         a = segs[i].get('text','')
#         b = segs[i+1].get('text','')
#         contradiction, detailed = run_pipeline(note_a=a, note_b=b, config_path=str(Path(repo_path)/'configs'/'config.yaml'), verbose=False, chunk_unit='sentences', chunk_size=1)
#         results.append({'i': i, 'contradiction': contradiction})
#     out_path = out_dir / (p.stem + '.diagnostic_shifts.json')
#     out_path.write_text(json.dumps({'note_id': p.stem, 'comparisons': results}, ensure_ascii=False, indent=2), encoding='utf-8')
# print('Batch run complete')